<a href="https://colab.research.google.com/github/CatalinaDeras24/CLAVE-G/blob/main/Notebook/Asociaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import pandas as pd

In [35]:
import warnings
warnings.filterwarnings("ignore")

#Cargar datos

In [36]:
url="https://raw.githubusercontent.com/CatalinaDeras24/CLAVE-G/refs/heads/main/data/clave_G_asociacion.csv"

In [37]:
df=pd.read_csv(url)

#Mostrar las primeras filas del dataset y explicar su estructura

In [38]:
print(df.head())


  transaccion_id cliente_id       fecha     categoria            item  \
0        G-T0001    G-C0072  2026-02-20  Construccion           Arena   
1        G-T0001    G-C0072  2026-02-20  Herramientas  Destornillador   
2        G-T0001    G-C0072  2026-02-20       Pintura            Lija   
3        G-T0002    G-C0057  2026-01-26  Electricidad           Cable   
4        G-T0002    G-C0057  2026-01-26  Electricidad     Interruptor   

   cantidad   canal  
0         1  Tienda  
1         1  Tienda  
2         3  Tienda  
3         2     App  
4         1     App  


In [39]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 622 entries, 0 to 621
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  622 non-null    object
 1   cliente_id      622 non-null    object
 2   fecha           622 non-null    object
 3   categoria       622 non-null    object
 4   item            622 non-null    object
 5   cantidad        622 non-null    int64 
 6   canal           621 non-null    object
dtypes: int64(1), object(6)
memory usage: 34.1+ KB
None


#Verificar valores nulos, duplicados y tipos de datos

In [40]:
print(df.isnull().sum())

transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64


In [41]:
df['canal'] = df['canal'].fillna('Desconocido')

In [42]:
print("Duplicados:", df.duplicated().sum())

Duplicados: 1


In [45]:
df = df.drop_duplicates()

In [46]:
print(df.dtypes)

transaccion_id    object
cliente_id        object
fecha             object
categoria         object
item              object
cantidad           int64
canal             object
dtype: object


#Preparar los datos para reglas de asociación

In [48]:
transacciones = df.groupby('transaccion_id')['item'].apply(list)

print(transacciones.head())

transaccion_id
G-T0001                        [Arena, Destornillador, Lija]
G-T0002    [Cable, Interruptor, Llave_paso, Martillo, Tom...
G-T0003                             [Arena, Brocha, Rodillo]
G-T0004                                [Llave_paso, Taladro]
G-T0005    [Arena, Cable, Cemento, Pegamento_PVC, Rodillo...
Name: item, dtype: object


In [49]:
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd


te = TransactionEncoder()
te_array = te.fit(transacciones).transform(transacciones)

df_binario = pd.DataFrame(te_array, columns=te.columns_)

print(df_binario.head())

   Alicate  Arena  Brocha  Cable  Cemento  Clavos   Codo  Destornillador  \
0    False   True   False  False    False   False  False            True   
1    False  False   False   True    False   False  False           False   
2    False   True    True  False    False   False  False           False   
3    False  False   False  False    False   False  False           False   
4    False   True   False   True     True   False  False           False   

    Foco  Interruptor   Lija  Llave_paso  Martillo  Pegamento_PVC  \
0  False        False   True       False     False          False   
1  False         True  False        True      True          False   
2  False        False  False       False     False          False   
3  False        False  False        True     False          False   
4  False        False  False       False     False           True   

   Pintura_blanca  Rodillo  Taladro  Tomacorriente  Tornillos  Tubo_PVC  
0           False    False    False          False    

#Aplicación del algoritmo Apriori

In [51]:
from mlxtend.frequent_patterns import apriori


frecuentes = apriori(
    df_binario,
    min_support=0.05,
    use_colnames=True
)

print(frecuentes.head())

    support   itemsets
0  0.097436  (Alicate)
1  0.102564    (Arena)
2  0.246154   (Brocha)
3  0.225641    (Cable)
4  0.123077  (Cemento)


#Generación de reglas de asociación

In [52]:
from mlxtend.frequent_patterns import association_rules


reglas = association_rules(
    frecuentes,
    metric='confidence',
    min_threshold=0.5
)


reglas = reglas.sort_values(by='lift', ascending=False)

print(reglas.head())

                 antecedents       consequents  antecedent support  \
9          (Rodillo, Brocha)  (Pintura_blanca)            0.102564   
8   (Pintura_blanca, Brocha)         (Rodillo)            0.112821   
7  (Pintura_blanca, Rodillo)          (Brocha)            0.123077   
2                 (Tubo_PVC)   (Pegamento_PVC)            0.205128   
3            (Pegamento_PVC)        (Tubo_PVC)            0.225641   

   consequent support   support  confidence      lift  representativity  \
9            0.241026  0.092308    0.900000  3.734043               1.0   
8            0.235897  0.092308    0.818182  3.468379               1.0   
7            0.246154  0.092308    0.750000  3.046875               1.0   
2            0.225641  0.128205    0.625000  2.769886               1.0   
3            0.205128  0.128205    0.568182  2.769886               1.0   

   leverage  conviction  zhangs_metric   jaccard  certainty  kulczynski  
9  0.067587    7.589744       0.815873  0.367347   0.8

#Mostrar las 10 reglas más relevantes



In [53]:
top10 = reglas[['antecedents',
                'consequents',
                'support',
                'confidence',
                'lift']].head(10)

print(top10)

                 antecedents       consequents   support  confidence      lift
9          (Rodillo, Brocha)  (Pintura_blanca)  0.092308    0.900000  3.734043
8   (Pintura_blanca, Brocha)         (Rodillo)  0.092308    0.818182  3.468379
7  (Pintura_blanca, Rodillo)          (Brocha)  0.092308    0.750000  3.046875
2                 (Tubo_PVC)   (Pegamento_PVC)  0.128205    0.625000  2.769886
3            (Pegamento_PVC)        (Tubo_PVC)  0.128205    0.568182  2.769886
1            (Tomacorriente)           (Cable)  0.112821    0.523810  2.321429
0                    (Cable)   (Tomacorriente)  0.112821    0.500000  2.321429
4           (Pintura_blanca)         (Rodillo)  0.123077    0.510638  2.164662
5                  (Rodillo)  (Pintura_blanca)  0.123077    0.521739  2.164662
6                  (Taladro)       (Tornillos)  0.087179    0.500000  2.031250
